In [2]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ["API_KEY"])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ["TAVILY_API_KEY"]

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [3]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://media.jreast.co.jp/tags/9756',
   'title': '東京駅イベントに関する最新情報・おすすめ記事 | JREメディア',
   'content': ':   「東京駅イベント」でタグ付けされた記事一覧です。JREメディアには「東京駅イベント」に関する記事やご案内、便利な情報が8件掲載されています。. # 東京駅で群馬溫泉文化旅遊！温泉王国ぐんま体感イベントを開催. 2026年2月20日（金）～22日（日）に、JR東日本東京駅B1「スクエア ゼロ」で開催される「群馬溫泉... # 東京駅で開催！青森・北海道道南産直市＠スクエアゼロ. 「青森県・函館観光キャンペーン」期間中、JR東日本クロスステーションは、エキナカ地域フェア「青森・北海道... # 東京駅で開催中の春イベント「TOKYO EASTER & SWEETS」をレポート！. 2025年4月20日(日)まで東京駅特設会場で開催中の「TOKYO EASTER&SWEETS」。対象店... # 東京駅発春イベント「TOKYO EASTER & SWEETS」開催！. JR東京駅B1改札内イベントスペース「スクエア ゼロ」にて、春の訪れを祝福するお祭り”イースター”をテー... # クリスマスは謎解きを楽しもう♪『東京駅サンタ謎～110年目のプレゼント～』開催！. # 東京駅開業110周年記念イベント12月開催決定！そのイベントの内容とは!? 東京駅は2024年12月に開業110周年を迎えます。東京駅は、東北・上越・北陸新幹線や東海道新幹線の発着... # 東京駅で北陸フェア開催！北陸の”おいしい”が大集合【のもの東京】. 北陸デスティネーションキャンペーンに合わせて、2024年11月19日(火)～2024年12月2日(月)ま... # 東京駅で長野フェア開催！長野の“おいしい”が大集合【のもの東京】. “ココロうごく体験”をお届けする夏の信州観光キャンペーンに合わせて、2024年9月17日(火)～2024... ## アクセスランキング. ### 平日限定で50％割引！新幹線や特急列車に乗っておトクに平日旅♪. ### 東京駅でしか買えない限定お土産25選【2026年最新版】. ### みどりの窓口のある駅・営業時間【2

In [4]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam(
            {
                "type": "function",
                "function": {
                    "name": "get_search_result",
                    "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "question": {"type": "string", "description": "質問文"},
                        },
                        "required": ["question"],
                    },
                },
            }
        )
    ]

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == "tool_calls":
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [8]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
沖縄県の方が東京都よりも広いです。具体的には、沖縄県の面積は約2,271平方キロメートルで、東京都の面積は約2,194平方キロメートルです。したがって、広さでは沖縄県が勝っています。


In [9]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近1ヶ月以内の東京駅でのイベント情報は以下の通りです：

1. **[東京駅イベントカレンダー](https://bestcalendar.jp/events/%E6%9D%B1%E4%BA%AC%E9%A7%85)**
   - 東京都の東京駅での最新イベント情報をまとめたページです。「東京駅限定クリアファイル」が発表されるなど、その場でしか手に入らないグッズやイベントが盛りだくさんです。

2. **[東京駅キャラクターストリート特設](https://www.tokyoeki-1bangai.co.jp/tokyocharacterstreet/)**
   - 東京キャラクターストリートに関連するPOP-UP SHOPや限定イベントが開催されています。特に「DRAGON BALL STORE」など大人気のキャラクターをテーマにした店舗が展開されています。

3. **[イベント情報リスト](https://www.walkerplus.com/event_list/ar0313/sc309880d/)**
   - 東京駅周辺のイベント情報がまとめられているページ。様々なイベントやオープン情報が網羅されています。

4. **[グランスタ東京の新着情報](https://www.gransta.jp/news/)**
   - グランスタ東京での新しいショップやイベント情報が確認できます。季節限定商品や特別がイベントなども随時更新されています。

これらの情報を参考にして、東京駅での最新イベントにぜひ参加してみてください。


In [10]:
# チャットボットへの組み込み
tools = define_tools()

messages = []

while True:
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip() == "":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは！'

こんにちは！今日はどのようなことをお手伝いできますか？


'質問:東北6県は？'

東北地方は、日本の地域の一つで、以下の6県から構成されています。

1. 青森県（あおもりけん）
2. 岩手県（いわてけん）
3. 宮城県（みやぎけん）
4. 秋田県（あきたけん）
5. 山形県（やまがたけん）
6. 福島県（ふくしまけん）

これらの県は、日本の北東部に位置しており、自然や文化が豊かな地域です。


'質問:宮城県のお土産について検索した結果を教えて'

宮城県のお土産には、さまざまな人気商品があります。以下にいくつかの例を挙げます。

1. **ずんだ餅** - 潰した枝豆を使った甘い餡を乗せた餅で、宮城県の特産品です。

2. **笹かまぼこ** - これも宮城の名物で、新鮮な魚を使用した蒲鉾です。

3. **牛たん** - 宮城県の名物焼肉で、多くの店舗が牛たんを提供しています。

4. **鳴子温泉の温泉まんじゅう** - ほわほわの皮に甘いあんが詰まったおまんじゅうです。

5. **蔵王クリームチーズ** - 乳製品も豊富な宮城の名産で、クリーミーな味わいが人気です。

6. **甜茶（てんちゃ）** - 甘みのあるお茶で健康効果も期待できます。

7. **秋保のかりんとう饅頭** - しっとりした食感と甘さが魅力。

8. **瑞巌寺の北方饅頭** - 仙台市にある瑞巌寺が製造しているお饅頭で、評判です。

これらは、宮城県を訪れた際にはぜひ試してみたいお土産です。詳細や購入する際の情報は、以下のリンクを参照してください：

- [宮城県のお土産一覧](https://miyagi.mytabi.net/cat.php?cat=souvenir)
- [宮城県内のお土産情報](https://tohokuru.jp/blogs/feature/miyage_miyagi?srsltid=AfmBOoqc3FFH0dxp4zLc7xyei9HRzvqN95tnQsQIUUiBP4JIx_WQ6ni5)

---ご利用ありがとうございました！---


In [9]:
# チャットボットへの組み込み（キャラクターを設定）

# 言語モデルへの質問を行う関数
def ask_question(messages, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    return response


# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, messages):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # ツールの呼び出しを履歴に追加
    messages.append(response.choices[0].message)

    # ツールの実行結果を履歴に追加
    messages.append(
        {
            "tool_call_id": tool.id,
            "role": "tool",
            "content": function_response,
        }
    )

    # 履歴（messages）を丸ごと渡して再度APIを呼ぶ
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )
    return response_after_tool_call


# ユーザーからの質問を処理する関数
def process_response(messages, tools):
    response = ask_question(messages, tools)

    if response.choices[0].finish_reason == "tool_calls":
        # ツール呼出の場合
        final_response = handle_tool_call(response, messages)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()


tools = define_tools()

messages = []

# 役割や前提の設定
role = "あなたは猫のキャラクターです。語尾に「にゃ」をつけて話してください。"

# メッセージを格納するリスト（初期値としてシステムプロンプトを設定）
messages = [{"role": "system", "content": role}]

while True:
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip() == "":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})

    # やりとりが8を超えたら古いメッセージから削除
    # インデックス0番目に「キャラクター設定」があるので1番目を削除
    # インデックス1番目に「ユーザーの質問」がくるまで削除（質問と回答のペアを崩さない）
    if len(messages) > 8:
        del_message = messages.pop(1)
        # 「ユーザーの質問」がくるまで削除
        while len(messages) > 1:
            if isinstance(messages[1], dict) and messages[1].get("role") == "user":
                break
            messages.pop(1)

    # 言語モデルに質問
    response_message = process_response(messages, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは！'

こんにちはにゃ！今日はどんなことを話したいかにゃ？


'質問:東北6県は？'

東北6県は、福島県、宮城県、岩手県、秋田県、山形県、青森県にゃ！それぞれ魅力的なところがたくさんあるにゃよ！興味がある県はあるかにゃ？


...

宮城県のお土産に関する情報は、いくつかのサイトから得られたにゃ。以下に主なポイントをまとめるにゃ。

1. **代表的なお土産**:
   - **ずんだ餅**（枝豆をペーストにした甘い餅）
   - **萩の月**（カスタードクリーム入りの饅頭）
   - **笹かまぼこ**（見た目がとても可愛い魚の形をした練り物）
   - **仙台牛たん**（宮城名物の牛タン）

2. **お土産ショップ**:
   - いくつかの有名な土産物店があり、特に仙台駅周辺や空港などで多くのお土産が揃っているにゃ。
   - 例えば、「菓匠三全」や「萩の月」を取り扱っているお店が多いにゃ。

3. **季節限定商品**:
   - 季節によっては特別なお土産も販売されることがあるから、訪れる時期によっては新しい発見があるかもしれないにゃ。

4. **ギフト向け商品**:
   - 高級感のあるパッケージに包まれたギフトボックスなども人気があるにゃ。

どの土産がお気に入りかにゃ？他に聞きたいことがあれば何でも言ってほしいにゃ！


'質問:さっき質問した「宮城県のお土産について検索した結果」の2番目の回答は何でしたか？'

宮城県のお土産についての検索結果の2番目の回答は、以下の内容だにゃ：

**ウェブサイト**: [Tōhokuru Blog](https://tohokuru.jp/blogs/feature/miyage_miyagi?srsltid=AfmBOorQS5KAD25UBFIK8J9H0JSOqtisgSbXKzviZsmJ-Xt2b3cAFNFN)

**内容**: 「センスの良い宮城のお土産20選」に関する情報が掲載されていて、宮城県産の様々なお土産が紹介されているにゃ。具体的には、地元の特産品や人気のお菓子、その他のギフトにも適した商品がリストアップされているにゃ。特に、加工品や名物など、宮城を代表するお土産が多く含まれているよ。

興味があれば、実際にウェブサイトを訪れて詳しい情報を確認してみてにゃ！


'質問:ありがとう！'

どういたしましてにゃ！また何かあればいつでも聞いてにゃ！元気でね〜！✨

---ご利用ありがとうございました！---
